Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 08 — system wyszukiwania podobnych obrazów (EuroSAT)

Materiał towarzyszący [`README.pl.md`](README.pl.md). Inny zbiór niż wykład (EuroSAT, nie
Caltech101) i inne zadanie: nie klasyfikacja, tylko **wyszukiwanie** — zapytanie
(obraz) → ranking najbardziej podobnych obrazów z bazy (gallery).


In [ ]:
import contextlib
import io
import os
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F401 - uzywane w sekcji 8 (wizualizacja)
import torch
import torch.nn as nn
import torchvision.datasets as tvd
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

## 1. Dane: gallery i queries


In [ ]:
_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = (
    Path(os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache"))
    .expanduser()
    .resolve()
)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    full_ds = tvd.EuroSAT(root=DATA_CACHE_DIR, download=True)

CLASSES = full_ds.classes
GALLERY_PER_CLASS = 40
QUERY_PER_CLASS = 10

class_to_indices = {c: [] for c in range(10)}
for idx, (_, label) in enumerate(full_ds):
    if len(class_to_indices[label]) < GALLERY_PER_CLASS + QUERY_PER_CLASS:
        class_to_indices[label].append(idx)
    if all(len(v) >= GALLERY_PER_CLASS + QUERY_PER_CLASS for v in class_to_indices.values()):
        break

gallery_indices, gallery_labels = [], []
query_indices, query_labels = [], []
for c in range(10):
    idxs = class_to_indices[c]
    gallery_indices += idxs[:GALLERY_PER_CLASS]
    gallery_labels += [c] * GALLERY_PER_CLASS
    query_indices += idxs[GALLERY_PER_CLASS : GALLERY_PER_CLASS + QUERY_PER_CLASS]
    query_labels += [c] * QUERY_PER_CLASS

print(f"gallery: {len(gallery_indices)} obrazow, queries: {len(query_indices)} obrazow")

In [ ]:
# TODO: zweryfikuj, ze gallery_indices i query_indices NIE maja wspolnych elementow
# (podpowiedz: set() i operator &)
raise NotImplementedError("TODO: weryfikacja rozlacznosci gallery/query")

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

gallery_imgs = torch.stack([preprocess(full_ds[i][0]) for i in gallery_indices])
query_imgs = torch.stack([preprocess(full_ds[i][0]) for i in query_indices])
gallery_y = torch.tensor(gallery_labels)
query_y = torch.tensor(query_labels)
print(f"gallery_imgs={tuple(gallery_imgs.shape)}  query_imgs={tuple(query_imgs.shape)}")

## 2. Ekstrakcja embeddingów


In [ ]:
import time

model = resnet18(weights=weights)
# TODO: ustaw model w tryb, ktory NIE aktualizuje statystyk BatchNorm
# (zobacz modul 7: requires_grad to NIE to samo co model.eval())
raise NotImplementedError("TODO: tryb modelu")

feat_extractor = nn.Sequential(*list(model.children())[:-1])

start = time.time()
# TODO: policz gallery_feats i query_feats (wyplaszczone wyjscie feat_extractor)
# POD kontekstem, ktory NIE buduje grafu autograd (torch.no_grad() albo
# torch.inference_mode() - zobacz modul 7 dla roznicy)
raise NotImplementedError("TODO: ekstrakcja embeddingow")
extract_time = time.time() - start
print(f"gallery_feats={tuple(gallery_feats.shape)}  czas: {extract_time:.2f}s")  # noqa: F821

**Zadanie:** gdyby baza była $1000\times$ większa, zapisałbyś/zapisałabyś embeddingi
na dysk. Pod jakimi **czterema** warunkami taki cache przestałby być poprawny?


_(Twoja odpowiedź tutaj.)_

## 3. Normalizacja L2


In [ ]:
def l2_normalize(X):
    # TODO: zwroc X podzielone przez norme KAZDEGO wiersza (z zabezpieczeniem
    # przed dzieleniem przez zero - zobacz wyklad)
    raise NotImplementedError("TODO: l2_normalize")


gallery_n = l2_normalize(gallery_feats)  # noqa: F821 - zdefiniowane w sekcji 2
query_n = l2_normalize(query_feats)  # noqa: F821 - zdefiniowane w sekcji 2
norms = gallery_n.norm(dim=1)
print(f"normy po normalizacji: min={norms.min():.4f} max={norms.max():.4f}")
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4)
print("OK: wszystkie embeddingi maja norme ~1 po normalizacji.")

## 4. Top-k wyszukiwanie

Napisz sam/sama ranking najbliższych sąsiadów wg cosine similarity — żadnej gotowej
funkcji bibliotecznej.


In [ ]:
def search(query_embedding, gallery_embeddings, k):
    """Zwraca indeksy k najbardziej podobnych wektorow z gallery (cosine
    similarity), posortowane od najlepszego. query_embedding: (d,).
    gallery_embeddings: (N, d)."""
    # TODO:
    # 1. znormalizuj query_embedding i gallery_embeddings (L2)
    # 2. policz cosine similarity miedzy query a KAZDYM wektorem w gallery
    #    (po normalizacji L2 to zwykly iloczyn skalarny)
    # 3. zwroc indeksy k najwiekszych wartosci, posortowane malejaco
    raise NotImplementedError("TODO: search")

**Weryfikacja na ręcznie skonstruowanym przykładzie z dokładnie znanym wynikiem:**


In [ ]:
toy_gallery = torch.tensor(
    [
        [1.0, 0.0],  # idx 0, klasa A
        [0.9, 0.1],  # idx 1, klasa A
        [0.8, 0.2],  # idx 2, klasa A
        [0.0, 1.0],  # idx 3, klasa B
        [0.1, 0.9],  # idx 4, klasa B
        [0.5, 0.5],  # idx 5, klasa A (dwuznaczny kierunek)
    ]
)
toy_gallery_y = torch.tensor([0, 0, 0, 1, 1, 0])
toy_query = torch.tensor([1.0, 0.0])  # dokladnie kierunek idx 0

toy_result = search(toy_query, toy_gallery, k=3)
print("wynik search():", toy_result.tolist())
expected = torch.tensor([0, 1, 2])  # policzone recznie: najwieksze cosine similarity
msg = f"oczekiwano {expected.tolist()}, dostano {toy_result.tolist()}"
assert torch.equal(toy_result, expected), msg
print("OK: search() zwraca dokladnie oczekiwany ranking na przykladzie kontrolnym.")

## 5. Precision@k


In [ ]:
def precision_at_k(query_embeddings, query_labels, gallery_embeddings, gallery_labels, k):
    """Srednie Precision@k po wszystkich zapytaniach. Trafnosc = zgodnosc klasy
    (uproszczenie - zobacz README)."""
    # TODO:
    # 1. dla kazdego zapytania: znajdz top-k w gallery (uzyj search())
    # 2. policz, jaki ulamek z tych k ma etykiete zgodna z etykieta zapytania
    # 3. usrednij po WSZYSTKICH zapytaniach (jawny mianownik: liczba zapytan)
    raise NotImplementedError("TODO: precision_at_k")

In [ ]:
toy_query_label = torch.tensor([0])
toy_precision = precision_at_k(
    toy_query.unsqueeze(0), toy_query_label, toy_gallery, toy_gallery_y, k=3
)
print("Precision@3 na przykladzie kontrolnym:", toy_precision)
assert toy_precision == 1.0, "oczekiwano 1.0 (wszystkie top-3 to klasa A)"
print("OK: precision_at_k() zgadza sie z recznym wyliczeniem.")

In [ ]:
for k in [1, 5, 10]:
    p = precision_at_k(query_n, query_y, gallery_n, gallery_y, k)
    print(f"Precision@{k} (ResNet18, cosine, znormalizowane): {p:.3f}")

## 6. Baseline: odległość pikseli


In [ ]:
# TODO: policz Precision@1/5/10 dla SUROWYCH, splaszczonych pikseli zamiast
# embeddingow ResNet18 (ta sama funkcja precision_at_k, ten sam gallery/query
# split - tylko inna reprezentacja wejsciowa, znormalizowana L2)
raise NotImplementedError("TODO: baseline pikseli")

## 7. Wpływ normalizacji i metryki


In [ ]:
# TODO:
# 1. wariant 1: cosine similarity na znormalizowanych L2 embeddingach (juz masz
#    z sekcji 5 - p_cosine_norm)
# 2. wariant 2: napisz dot_search() - ranking wg zwyklego iloczynu skalarnego
#    na NIEznormalizowanych gallery_feats/query_feats (bez l2_normalize)
# 3. wariant 3: napisz euclidean_search() - ranking wg odleglosci euklidesowej
#    na NIEznormalizowanych gallery_feats/query_feats
# 4. policz Precision@5 dla wariantow 2 i 3 (mozesz napisac analogiczne
#    funkcje precision_at_k_dot/precision_at_k_euclidean, albo uogolnic search())
# 5. wypisz wszystkie trzy wyniki obok siebie
raise NotImplementedError("TODO: wplyw metryki i normalizacji")

_(Twoja odpowiedź tutaj.)_

## 8. Wizualizacja i analiza błędów


In [ ]:
# TODO:
# 1. napisz funkcje show_query_and_results(query_idx_in_query_set, k=5), ktora
#    rysuje zapytanie i jego top-k wynikow z gallery (obraz + etykieta + czy
#    trafiono) - pamietaj o odwroceniu normalizacji preprocessingu do wyswietlenia
# 2. policz Precision@5 OSOBNO dla kazdego zapytania (nie usrednione), znajdz
#    zapytanie z najwyzszym i najnizszym wynikiem
# 3. wywolaj show_query_and_results() dla obu
# Rozszerzenie opcjonalne: policz systematycznie po WSZYSTKICH zapytaniach,
# ktore pary (prawdziwa klasa, znaleziona klasa) pojawiaja sie najczesciej
# wsrod bledow w top-5
raise NotImplementedError("TODO: wizualizacja i analiza bledow")

_(Twoja odpowiedź tutaj.)_

## 9. Wnioski

Odpowiedz, uzasadniając rzeczywistymi liczbami z sekcji 5-8.


**1. O ile embeddingi ResNet18 przebijają baseline surowych pikseli? Czy to zaskakujące, biorąc pod uwagę domenę odległą od ImageNet?**

_(Twoja odpowiedź tutaj.)_

**2. Czy wybór metryki/normalizacji miał duży wpływ? Normalizowałbyś/łabyś embeddingi, budując taki system od zera?**

_(Twoja odpowiedź tutaj.)_

**3. Zgodność etykiety to uproszczona miara trafności. Znalazłaś/eś przykład, gdzie to uproszczenie zawodzi?**

_(Twoja odpowiedź tutaj.)_

**4. Jakie jest co najmniej jedno ograniczenie tego eksperymentu, które nie pozwala uogólnić wniosków na dowolny system wyszukiwania obrazów?**

_(Twoja odpowiedź tutaj.)_